In [17]:
import os

matches = []

for root, dirs, files in os.walk(r"C:\Users\VIJAY"):
    if "openai_key.txt" in files:
        matches.append(os.path.join(root, "openai_key.txt"))

print(matches)

['C:\\Users\\VIJAY\\openai_key.txt']


In [18]:
import re
from openai import OpenAI
raw = open(r"C:\Users\VIJAY\openai_key.txt", "r").read()
m = re.search(r"(sk-[A-Za-z0-9\-_]+)", raw)
if not m:
    raise RuntimeError("No API key found in openaikey.txt — put the raw key or adjust parsing.")
api_key = m.group(1)
client = OpenAI(api_key=api_key)
resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role":"user","content":"ping"}]
)
print(resp.choices[0].message.content)

Pong! How can I assist you today?


In [4]:
import PyPDF2
import faiss
import numpy as np
from openai import OpenAI


In [26]:
def read_pdf(path):
    reader=PyPDF2.PdfReader(path)
    text=""
    for page in reader.pages:
        page_text=page.extract_text()
        if page_text:
            text=text+page_text
    return text
text=read_pdf(r"sensors.pdf")
print(text)
print(len(text))
    

Citation: Alatoun, K.; Matrouk, K.;
Mohammed, M.A.; Nedoma, J.;
Martinek, R.; Zmij, P . A Novel
Low-Latency and Energy-Efﬁcient
Task Scheduling Framework for
Internet of Medical Things in an Edge
Fog Cloud System. Sensors 2022 ,22,
5327. https://doi.org/10.3390/
s22145327
Academic Editor: Paolo Bellavista
Received: 18 June 2022
Accepted: 13 July 2022
Published: 16 July 2022
Publisher’s Note: MDPI stays neutral
with regard to jurisdictional claims in
published maps and institutional afﬁl-
iations.
Copyright: © 2022 by the authors.
Licensee MDPI, Basel, Switzerland.
This article is an open access article
distributed under the terms and
conditions of the Creative Commons
Attribution (CC BY) license (https://
creativecommons.org/licenses/by/
4.0/).
sensors
Article
A Novel Low-Latency and Energy-Efﬁcient Task Scheduling
Framework for Internet of Medical Things in an Edge Fog
Cloud System
Kholoud Alatoun1, Khaled Matrouk1
, Mazin Abed Mohammed2, Jan Nedoma3,*
, Radek Martinek4
and Petr Zmij5

In [6]:
def make_chunks(text,chunk_size=500):
    chunklist=[]
    start=0
    while start<len(text):
        end=start+chunk_size
        chunk=text[start:end]
        chunklist.append(chunk)
        start=end
    return chunklist
cl=make_chunks(text)
print(len(cl))

244


In [7]:
def get_embeddings(chunks):
    embeddings = []

    for chunk in chunks:
        response = client.embeddings.create(input = chunk ,model="text-embedding-3-small" )
        vector = response.data[0].embedding
        embeddings.append(vector)
    return np.array(embeddings).astype("float32")

In [8]:
def build_faiss_index(embeddings):
    dimension = len(embeddings[0]) 
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)
    return index 

In [9]:
def search_chunks(question, chunks , index):
    q_embed = client.embeddings.create(input=question,model="text-embedding-3-small").data[0].embedding
    q_embed = np.array([q_embed]).astype("float32")
    distaces , indices = index.search(q_embed,k=2)
    best_chunks = [chunks[i] for i in indices[0]]
    return best_chunks

In [22]:
def ask_question(chunks, question):
    context = "\n\n".join(chunks)  
    prompt = f"Answer the question based on this text:\n{context}\n\nQuestion: {question}"
    response = client.chat.completions.create(
        model="gpt-4o-mini", 
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

In [29]:
em = get_embeddings(cl)
index = build_faiss_index(em)

In [28]:
question = input("Ask your question: ")
best_chunks = search_chunks(question, cl, index)
answer = ask_question(best_chunks, question)
print(answer)

Ask your question:  What is a sensor


A sensor is a device that detects and measures physical properties, such as physiological data and vital signals. In the context of healthcare, sensors are involved in sensing processes and can be part of a system that collects data for applications like remote telecare. They play a key role in monitoring health parameters by gathering information about the patient's condition, which is essential for timely and effective healthcare delivery.
